In [1]:
#Add path to recording here
RecordingsToUse = [r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2024\082324\J705']

Contam_Threshold = 30




In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statistics as stat 
import math 
from pylab import *
import os
import json
from tqdm import tqdm

#########################################################################
#Autocorrelogram function 

def autocorrelogram(SpikeT):
    ISI = np.empty(0)
    for ii in range(0,np.size(SpikeT)):

        Spike = SpikeT[ii]      #Spike to compare to others 

        SpikeMask = SpikeT[np.argwhere(np.abs(SpikeT-Spike)<= .05)]      #Find only spikes within +- .05 sec of spike of interest, to save on computations
        TempSpikes = np.delete(SpikeMask,np.argwhere(SpikeMask == Spike)[0][0])     #Remove spike of interest (Spike), so no peak at 0 in correlelogram

        TempSpikes = TempSpikes - Spike     #Calculating ISI for spikes within +-.05 secs

        ISI = np.concatenate((ISI,TempSpikes))
    return ISI



#########################################################################
#Bins for Histograms

contam_bins = np.arange(0,300,5)
fr_bins = np.arange(0,20,.5)
autocor_bins = np.arange(-.01,.01,.0002)


#########################################################################

In [4]:
for zz in list(range(0,len(RecordingsToUse))):
    Dk = 0

    RecordingPath = RecordingsToUse[zz]
    #########################################################################
    #Finding FM ephys files before DOI 

    SavePath =  RecordingPath + "/Analysis/Single Unit Plots" 


    if not os.path.exists(SavePath):
        os.makedirs(SavePath)

    Items = os.listdir(RecordingPath)
    first_wn = 0
    for names in Items:
        if names.endswith("cluster_info.tsv"):
            print('cluster info data found')
            print(os.path.join(RecordingPath,names))
            cluster_info= pd.read_csv(os.path.join(RecordingPath,names),sep = '\t',index_col=0)

        if names.endswith("templates.npy"):
            if len(names) == 13:
                template = np.load(os.path.join(RecordingPath,names))
                print('template data found')

        if names.endswith('wn'):
            if first_wn == 0:
                WN_EphysFiles = []
                wn_folder = names
                for files in os.listdir(os.path.join(RecordingPath, wn_folder)):
                    if files.endswith("ephys_props.h5"):
                        WN_EphysFiles = files
                first_wn = 1 

        if names.endswith(('fm1','fm1-darkness','fm1_dark','fm1_darkness')):
            FM1_Path = os.path.join(RecordingPath, names)
            for files in os.listdir(FM1_Path):
                if files.endswith('ephys_props.h5'):
                    print('fm1 data found in ', FM1_Path)
                    PreDOI_InputData = os.path.join(FM1_Path, files)

        if names.endswith('fm2'):
            FM2_Path = os.path.join(RecordingPath, names)
            for files in os.listdir(FM2_Path):
                if files.endswith('ephys_props.h5'):
                    print('fm2 data found in ', FM2_Path)
                    PostDOI_InputData = os.path.join(FM2_Path, files)

    #Pulling Variables of Interest
    PreDOI_Data = pd.read_hdf(PreDOI_InputData)
    PostDOI_Data = pd.read_hdf(PostDOI_InputData)
    try:
        WN_EphysFiles
    except NameError:
        No_WN_Data = 1
    else:
        No_WN_Data = 0
        WN_EphysData = pd.read_hdf(os.path.join(RecordingPath, wn_folder, WN_EphysFiles))

    #FiringRate = PostDOI_Data.loc[:,'FmLt_fr']
    try:
        FiringRate = PreDOI_Data.loc[:,'FmLt_fr']
        ContamPct = PreDOI_Data.loc[:,'FmLt_ContamPct']
    except:
        FiringRate = PreDOI_Data.loc[:,'FmDk_fr']
        ContamPct = PreDOI_Data.loc[:,'FmDk_ContamPct']
        Dk = 1
        print('dark data used')

    ResponsiveClusters = FiringRate[(FiringRate >= .5)].index

    #ContamPct = PostDOI_Data.loc[:,'FmLt_ContamPct']
    
    NotContaminatedClusters = ContamPct[(ContamPct <= Contam_Threshold)].index
    HighContaminatedThresh = ContamPct[(ContamPct <= 25 )].index

    GoodButContaminatedClusters = np.intersect1d(ResponsiveClusters,HighContaminatedThresh)
    GoodClusters = np.intersect1d(ResponsiveClusters,NotContaminatedClusters)

    #########################################################################     

    print(len(NotContaminatedClusters), " Clusters below Contamination Threshold")
    print( )
    print(len(ResponsiveClusters), " Clusters with Firing Rates >= 0.5Hz")
    print( )
    print(len(GoodClusters), " Clusters with Firing Rates >= 0.5Hz and Contamination Index below 30% Contamination Threshold")
    print( )
    print(len(GoodButContaminatedClusters), " Clusters with Firing Rates >= 0.5Hz and Contamination Index below 25% Contamination Threshold")

    #########################################################################    
    #Looping through each cluster in dataset(ignoring contamination Idx) and making/saving a figure of eye movement PSTHs before and after DOI

    for ii in tqdm(list(range(0,len(GoodClusters)))):
        
        Cell = GoodClusters[ii]

        title = 'Cluster ' + str(Cell) + ' Index ' + str(ii)
        
        fig, axs = plt.subplots(3,2, figsize=(10,10))
        fig.suptitle(title)

        if Dk == 1:
            axs[0,0].plot(PreDOI_Data.loc[Cell,'FmDk_waveform'],color = 'red',label='pipeline')
            try:
                OG_Idx = int(np.squeeze(np.where((PreDOI_Data.loc[Cell]['FmDk_Amplitude'] == cluster_info['Amplitude']) & 
                (np.round(PreDOI_Data.loc[Cell]['FmDk_ContamPct'],6) == cluster_info['ContamPct']))))
                
            except:
                OG_Idx = int(np.squeeze(np.where((PreDOI_Data.loc[Cell]['FmDk_Amplitude'] == cluster_info['Amplitude']) & 
                (np.round(PreDOI_Data.loc[Cell]['FmDk_ContamPct'],6) == cluster_info['ContamPct']) &
                (PreDOI_Data.loc[Cell]['FmDk_amp'] == cluster_info['amp'])))) 
        else:
            axs[0,0].plot(PreDOI_Data.loc[Cell,'FmLt_waveform'],color = 'red',label='pipeline')
            try:
                OG_Idx = int(np.squeeze(np.where((PreDOI_Data.loc[Cell]['FmLt_Amplitude'] == cluster_info['Amplitude']) & 
                (np.round(PreDOI_Data.loc[Cell]['FmLt_ContamPct'],6) == cluster_info['ContamPct']))))
                
            except:
                OG_Idx = int(np.squeeze(np.where((PreDOI_Data.loc[Cell]['FmLt_Amplitude'] == cluster_info['Amplitude']) & 
                (np.round(PreDOI_Data.loc[Cell]['FmLt_ContamPct'],6) == cluster_info['ContamPct']) &
                (PreDOI_Data.loc[Cell]['FmLt_amp'] == cluster_info['amp']))))

        
        try:
            WaveformKS25 = template[OG_Idx,:,cluster_info.iloc[OG_Idx]['ch']]
            WaveformKS25_Norm = WaveformKS25/np.nanmax(np.abs(WaveformKS25))
            axs[0,0].plot(WaveformKS25_Norm,color = 'blue',label='KS25')
        except:
            try:
                WaveformKS4 = template[cluster_info.index[OG_Idx],:,cluster_info.iloc[OG_Idx]['ch']]
                WaveformKS4_Norm = WaveformKS4/np.nanmax(np.abs(WaveformKS4))
                axs[0,0].plot(WaveformKS4_Norm,color = 'black',label='KS4')
            except:
                dummy = 0
                
        axs[0,0].legend()
        if No_WN_Data == 0:
            try:
                axs[0,1].imshow(WN_EphysData.loc[Cell,'Wn_spike_triggered_average'])
            except:
                print('No WN RF Data')

        if Dk == 1:
            PreDOI_autocor = autocorrelogram(PreDOI_Data.loc[Cell, 'FmDk_spikeT'])
            PostDOI_autocor = autocorrelogram(PostDOI_Data.loc[Cell, 'FmLt_spikeT'])

            axs[2,0].plot(PreDOI_Data.loc[Cell,'FmDk_gazeshift_rightPSTH'],color='b')
            axs[2,0].plot(PreDOI_Data.loc[Cell,'FmDk_gazeshift_leftPSTH'],color='r')

            axs[2,1].plot(PostDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'],color='b')
            axs[2,1].plot(PostDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'],color='r')
        else:
            PreDOI_autocor = autocorrelogram(PreDOI_Data.loc[Cell, 'FmLt_spikeT'])
            PostDOI_autocor = autocorrelogram(PostDOI_Data.loc[Cell, 'FmLt_spikeT'])

            axs[2,0].plot(PreDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'],color='b')
            axs[2,0].plot(PreDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'],color='r')

            axs[2,1].plot(PostDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'],color='b')
            axs[2,1].plot(PostDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'],color='r')


        axs[1,0].hist(PreDOI_autocor,autocor_bins,color = 'black')
        axs[1,0].set_xlabel('Pre DOI autocorrelogram (s)')
        axs[1,0].axvline(.001,linestyle='--',color='red')
        axs[1,0].axvline(-.001,linestyle='--',color='red')

        #Plotting Pre DOI autocorrelogram
        axs[1,1].hist(PostDOI_autocor,autocor_bins,color = 'lime')
        axs[1,1].set_xlabel('Post DOI autocorrelogram (s)')
        axs[1,1].axvline(.001,linestyle='--',color='red')
        axs[1,1].axvline(-.001,linestyle='--',color='red')

        axs[2,0].set_ylabel('Gaze Shifts')
        axs[2,0].legend("RL",loc= "upper left")
        
        plt.savefig(SavePath + "/Cell " +str(Cell) + '.png',bbox_inches='tight')
        plt.close(fig)


    np.save(SavePath + '/HighContaminatedThresh_Temp', HighContaminatedThresh)
    np.save(SavePath + '/GoodClusters_Temp', GoodClusters)



cluster info data found
\\goeppert\Vol2\Rolf\FreelyMovingEphys\2024\082324\J705\cluster_info.tsv
fm1 data found in  \\goeppert\Vol2\Rolf\FreelyMovingEphys\2024\082324\J705\fm1
fm2 data found in  \\goeppert\Vol2\Rolf\FreelyMovingEphys\2024\082324\J705\fm2
template data found
159  Clusters below Contamination Threshold

227  Clusters with Firing Rates >= 0.5Hz

139  Clusters with Firing Rates >= 0.5Hz and Contamination Index below 10% Contamination Threshold

127  Clusters with Firing Rates >= 0.5Hz and Contamination Index below 25% Contamination Threshold


100%|██████████| 139/139 [1:29:38<00:00, 38.69s/it]  


In [10]:
#After looking through plots created above list any cells to be removed below
ToRemove= np.array([2,5,16,26,36,47,49,60,70,75,93,112,135,136,137,139,141,146,172,178,207,227,228,246,250,255,262,265,302,315,328,340,344])



In [11]:
SingleUnits_HighThresh = np.setdiff1d(HighContaminatedThresh, ToRemove)
SingleUnits = np.setdiff1d(GoodClusters, ToRemove)

print(len(SingleUnits_HighThresh), " Single Units high Threshold  of Contamination")
print(len(SingleUnits), " Not Contaminated Single Units")


np.save(RecordingPath + '/SingleUnits_HighThresh', SingleUnits_HighThresh)
np.save(RecordingPath + '/SingleUnits', SingleUnits)
np.save(RecordingPath + '/RemovedClusters', ToRemove)

99  Single Units high Threshold  of Contamination
47  Not Contaminated Single Units


#Old 


#Finding FM ephys files before DOI 

Items = os.listdir(RecordingPath)

for names in Items:
    if names.endswith("cluster_info.tsv"):
        print('cluster info data found')
        cluster_info= pd.read_csv(os.path.join(RecordingPath,names),sep = '\t',index_col=0)

    if names.endswith("templates.npy"):
        if len(names) == 13:
            template = np.load(os.path.join(RecordingPath,names))
            print('template data found')

    if names.endswith('wn'):
        WN_EphysFiles = []
        wn_folder = names
        for files in os.listdir(os.path.join(RecordingPath, wn_folder)):
            if files.endswith("ephys_merge.json"):
                WN_EphysFiles = files
                WN_EphysData = pd.read_json(os.path.join(RecordingPath, wn_folder, WN_EphysFiles))

    if names.endswith('fm1'):
        FM1_Path = os.listdir(os.path.join(RecordingPath, names))
        for Items in os.listdir(FM1_Path):
            PreDOI_InputData = os.path.join(FM1_Path, Items)

    if names.endswith('fm2'):
        FM2_Path = os.listdir(os.path.join(RecordingPath, names))
        for Items in os.listdir(FM2_Path):
            PostDOI_InputData = os.path.join(FM2_Path, Items)

#Pulling Variables of Interest
PreDOI_Data = pd.read_hdf(PreDOI_InputData)
PostDOI_Data = pd.read_hdf(PostDOI_InputData)

#FiringRate = PostDOI_Data.loc[:,'FmLt_fr']
FiringRate = PreDOI_Data.loc[:,'FmLt_fr']
ResponsiveClusters = FiringRate[(FiringRate >= .5)].index

#ContamPct = PostDOI_Data.loc[:,'FmLt_ContamPct']
ContamPct = PreDOI_Data.loc[:,'FmLt_ContamPct']
NotContaminatedClusters = ContamPct[(ContamPct <= Contam_Threshold)].index
HighContaminatedThresh = ContamPct[(ContamPct <= 25 )].index

GoodClusters = np.intersect1d(ResponsiveClusters,NotContaminatedClusters)

#########################################################################     

print(len(NotContaminatedClusters), " Clusters below Contamination Threshold")
print( )
print(len(ResponsiveClusters), " Clusters with Firing Rates >= 0.5Hz")
print( )
print(len(GoodClusters), " Clusters with Firing Rates >= 0.5Hz and Contamination Index below Contamination Threshold")

#########################################################################    
#Looping through each cluster in dataset(ignoring contamination Idx) and making/saving a figure of eye movement PSTHs before and after DOI

SavePath =  RecordingPath + "/Analysis/Single Unit Plots" 

if not os.path.exists(SavePath):
    os.makedirs(SavePath)


Chnl = PostDOI_Data.loc[:,'FmLt_ch']
#Chnl = PreDOI_Data.loc[:,'FmLt_ch']

for ii in tqdm(list(range(0,len(GoodClusters)))):
    
    Cell = GoodClusters[ii]
    
    title = 'Cluster ' + str(Cell) + ' Index ' + str(ii)
    
    fig, axs = plt.subplots(3,2, figsize=(10,10))
    fig.suptitle(title)


    yTicks = []
    yLabels = []
    Ys = 0
    Waveform_Data = template[Cell,:,:]
    Waveform_Data = Waveform_Data/np.nanmax(np.abs(Waveform_Data))

    for cnl in range(0,128):
        if len(np.unique(Waveform_Data[:,cnl])) > 1:
            yLabels = np.append(yLabels, cnl)
            if cnl == Chnl[Cell]:
                axs[0,0].plot(Waveform_Data[:,cnl]+Ys,color = 'blue')
            else:
                axs[0,0].plot(Waveform_Data[:,cnl]+Ys,color = 'black')
            Ys = Ys + .1 
    if len(yLabels) == len(np.arange(0,Ys,.1)):
        axs[0,0].set_yticks(ticks = np.arange(0,Ys,.1),labels=yLabels)  
    else:
        axs[0,0].set_yticks(ticks = np.arange(0,Ys-.09,.1),labels=yLabels)

    

    #Plotting Spike Rate (Spikes/min) Histogram 
    #axs[0,1].hist((PreDOI_Data.loc[Cell,'FmDk_spikeT']/60),np.arange(0,60,1),histtype='step',color='black')
    axs[0,1].hist((PreDOI_Data.loc[Cell,'FmLt_spikeT']/60),np.arange(0,60,1),histtype='step',color='black')
    axs[0,1].hist((PostDOI_Data.loc[Cell,'FmLt_spikeT']/60),np.arange(0,60,1),histtype='step',color='green')
    axs[0,1].title.set_text('Spikes/Min Across FM1 (b) and FM2 (r)')
 
  #  #Plotting Contamination Perecetange Histogram
  #  axs[1,1].hist(ContamPct,contam_bins)
  #  axs[1,1].axvline(Contam_Threshold,color = 'black')
  #  axs[1,1].axvline(PreDOI_Data.loc[Cell,'FmLt_ContamPct'],color = clr)
  #  axs[1,1].title.set_text('Contamination Percentage')


    #Plotting GazeShift_leftPSTH and _rightPSTH before DOI
    #axs[2,0].plot(PreDOI_Data.loc[Cell,'FmDk_gazeshift_rightPSTH'],color='b')
    axs[2,0].plot(PreDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'],color='b')
    #axs[2,0].plot(PreDOI_Data.loc[Cell,'FmDk_gazeshift_leftPSTH'],color='r')
    axs[2,0].plot(PreDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'],color='r')
    axs[2,0].set_ylabel('Gaze Shifts')
    axs[2,0].legend("RL",loc= "upper left")

   #Plotting GazeShift_leftPSTH and _rightPSTH after DOI
    axs[2,1].plot(PostDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'],color='b')
    axs[2,1].plot(PostDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'],color='r')


    #Plotting Pre DOI autocorrelogram
    #PreDOI_autocor = autocorrelogram(PreDOI_Data.loc[Cell, 'FmDk_spikeT'])
    PreDOI_autocor = autocorrelogram(PreDOI_Data.loc[Cell, 'FmLt_spikeT'])
    axs[1,0].hist(PreDOI_autocor,autocor_bins,color = 'black')
    axs[1,0].set_xlabel('Pre DOI autocorrelogram (s)')
    axs[1,0].axvline(.001,linestyle='--',color='red')
    axs[1,0].axvline(-.001,linestyle='--',color='red')
    #Plotting Pre DOI autocorrelogram
    PostDOI_autocor = autocorrelogram(PostDOI_Data.loc[Cell, 'FmLt_spikeT'])
    axs[1,1].hist(PostDOI_autocor,autocor_bins,color = 'lime')
    axs[1,1].set_xlabel('Post DOI autocorrelogram (s)')
    axs[1,1].axvline(.001,linestyle='--',color='red')
    axs[1,1].axvline(-.001,linestyle='--',color='red')

    plt.savefig(SavePath + "/Cell " +str(Cell) + '.png',bbox_inches='tight')
    plt.close(fig)
    
#########################################################################   
#Plotting Contamination Index with Cutoff Value of 15


fig, axs = plt.subplots(1,2)
axs[0].hist(ContamPct,contam_bins)
axs[0].axvline(Contam_Threshold,color='black')
axs[0].set_title('Contamination Index')
axs[0].text(40,0,len(NotContaminatedClusters),color = 'red')

axs[1].hist(FiringRate,fr_bins)
axs[1].axvline(.5,color='black')
axs[1].set_title('Firing Rate')
axs[1].text(5,0,len(ResponsiveClusters),color = 'red')

del PreFM_Path, PostFM_Path